# example_training_plots — CPU walkthrough

This notebook runs the same end-to-end MLflow workflow as
`experiments/example_training_plots.py`, but split into small cells with
explanations so you can re-run individual steps and inspect intermediate state.

**What you'll do here**
1. Connect to the MLflow tracking server (URI from `.env`).
2. Open a run and stream a 600-step *simulated* training trajectory.
3. Register the training history as an MLflow **Dataset** entity.
4. Fit a real (small) sklearn `Ridge` model on the history, log it, and
   **register** it in the Model Registry under a global name.
5. Generate and upload the full plot bundle (loss / LR / grad-norm / etc.).

The training data here is synthetic — the goal is to exercise the tracking,
artifact, and registry plumbing, not to actually pretrain a language model.


## Naming conventions you must keep straight

MLflow has **two distinct names** for a model — confusing them is the most
common reason "the model didn't show up where I expected":

| Name | Where it lives | Scope |
|---|---|---|
| `artifact_path` (a.k.a. `name=` in MLflow 3) | The run's **Artifacts** tab, at `runs:/<run_id>/<artifact_path>` | Just this run |
| `registered_model_name` | The **Models** tab in the registry | Global; each call adds a *new version* |

If you only set `artifact_path`, the model is logged but **not** registered.
If you also set `registered_model_name`, MLflow logs *and* creates a version.


In [ ]:
# Local artifact name (visible inside this run only):
MODEL_ARTIFACT_PATH = "loss-predictor"

# Global name in the Model Registry (visible across runs in the Models tab):
REGISTERED_MODEL_NAME = "burnit-bg-loss-predictor"

# Alias attached to the new version. Aliases can be moved later
# (e.g. promote the best run from 'staging' to 'production').
MODEL_ALIAS = "staging"
SOURCE_ARTIFACT_PATH = "notebooks"


## Helpers

The next three cells define the same helpers as the `.py` script:

* `cosine_lr_schedule` — the standard *linear warmup → cosine decay* used by
  most LLM pretraining recipes.
* `simulate_training` — produces a realistic-looking 600-step trajectory
  (loss, LR, grad-norm, throughput) plus periodic eval metrics.
* `fake_attention_matrix` — a synthetic causal-attention pattern used for
  the attention heatmap plot.


In [ ]:
def cosine_lr_schedule(step, total, warmup, peak_lr, min_lr):
    """Linear warmup -> cosine decay. The standard LLM pretraining schedule."""
    import math
    if step < warmup:
        return peak_lr * (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def simulate_training(*, total_steps=600, warmup_steps=60, eval_every=50,
                      peak_lr=3e-4, min_lr=1e-5, seed=42):
    """Generate plausible training and eval metric series.

    Returns a dict of numpy arrays: train/eval loss, perplexity, learning rate,
    grad norms (with occasional spikes), throughput (tokens/sec), and step time.
    """
    rng = np.random.default_rng(seed)
    steps = np.arange(1, total_steps + 1)

    # train loss: exponential decay + small sinusoidal drift + noise, clipped at 1.2
    base = 7.5 * np.exp(-steps / 220.0) + 1.6
    drift = 0.05 * np.sin(steps / 35.0)
    noise = rng.normal(0.0, 0.04, size=total_steps)
    train_loss = np.clip(base + drift + noise, 1.2, None)

    eval_steps = steps[steps % eval_every == 0]
    eval_loss = np.array([train_loss[s - 1] + 0.18 + rng.normal(0.0, 0.03) for s in eval_steps])

    eval_precision, eval_recall, eval_f1, eval_accuracy = [], [], [], []
    for loss in eval_loss:
        n = 1500
        y_true = rng.integers(0, 2, size=n)
        sep = float(np.clip((8.0 - loss) / 8.0, 0.10, 0.90))
        logits = (2 * y_true - 1) * sep + rng.normal(0.0, 0.95, size=n)
        y_pred = (logits > 0.0).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        eval_precision.append(float(p))
        eval_recall.append(float(r))
        eval_f1.append(float(f))
        eval_accuracy.append(float(accuracy_score(y_true, y_pred)))

    train_ppl = np.exp(train_loss)
    eval_ppl = np.exp(eval_loss)

    lr = np.array([cosine_lr_schedule(s - 1, total_steps, warmup_steps, peak_lr, min_lr)
                   for s in steps])

    grad_norm = np.abs(rng.normal(0.6, 0.18, size=total_steps))
    grad_norm[:warmup_steps] *= 1.6
    spike_idx = rng.choice(total_steps // 4, size=4, replace=False)
    grad_norm[spike_idx] *= rng.uniform(3.5, 5.5, size=spike_idx.size)

    base_tps = 6500.0 + 700.0 * (1.0 - np.exp(-steps / 50.0))
    jitter = rng.normal(0.0, 200.0, size=total_steps)
    tokens_per_sec = np.clip(base_tps + jitter, 3000.0, None)

    batch_tokens = 8192
    step_seconds = batch_tokens / tokens_per_sec

    return {
        "steps": steps, "train_loss": train_loss, "eval_steps": eval_steps,
        "eval_loss": eval_loss, "eval_precision": np.array(eval_precision),
        "eval_recall": np.array(eval_recall), "eval_f1": np.array(eval_f1),
        "eval_accuracy": np.array(eval_accuracy), "train_ppl": train_ppl,
        "eval_ppl": eval_ppl, "learning_rate": lr, "grad_norm": grad_norm,
        "tokens_per_sec": tokens_per_sec, "step_seconds": step_seconds,
    }


In [ ]:
def fake_attention_matrix(n_tokens=16, seed=7):
    """Synthetic causal attention pattern with a soft diagonal bias."""
    rng = np.random.default_rng(seed)
    raw = rng.gamma(1.5, 1.0, size=(n_tokens, n_tokens))
    raw = np.tril(raw)
    for i in range(n_tokens):
        raw[i, max(0, i - 2): i + 1] *= 3.0
    raw = np.exp(raw - raw.max(axis=1, keepdims=True))
    raw = raw * np.tri(n_tokens)
    raw = raw / raw.sum(axis=1, keepdims=True).clip(1e-12)
    return raw


## Step 1 — Load env, connect to MLflow

`set_env()` reads `.env` from the repo root. `MLflowTracking.from_env()`
constructs the wrapper using `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME`.

`check_connection()` does a `search_experiments()` call so any auth/TLS
problem surfaces immediately (rather than at first metric write).

In [ ]:
from data_platform.common import set_env
from data_platform.tracking import MLflowTracking

set_env()
tracking = MLflowTracking.from_env()
tracking.check_connection()


## Step 2 — Define the run config

`run_tags` are key-value labels searchable in the UI; `run_params` are the
hyperparameters that uniquely identify this experiment. Both are fully logged.

In [ ]:
total_steps = 600
warmup_steps = 60
eval_every = 50

run_tags = {"task": "pretraining-sim", "framework": "synthetic"}
run_params = {
    "total_steps": total_steps,
    "warmup_steps": warmup_steps,
    "eval_every": eval_every,
    "batch_tokens": 8192,
    "peak_lr": 3e-4,
    "min_lr": 1e-5,
    "optimizer": "adamw",
    "scheduler": "cosine",
    "primary_metric": "eval_f1",
}


## Step 3 — Open the MLflow run

In the `.py` script we used `with tracking.run(...) as run:`. In a notebook
that's awkward because the `with` block can't span cells. Instead we call
`mlflow.start_run(...)` directly and `mlflow.end_run()` at the bottom — every
cell in between operates against the active run.

We also call `tracking.log_hardware(step=0)` to record an initial CPU/RAM/GPU
snapshot (matches what the context manager does automatically).

In [ ]:
import mlflow
from pathlib import Path

# Resolve the path of this notebook (works in JupyterLab/VSCode notebooks).
NOTEBOOK_PATH = Path.cwd() / "example_training_plots.ipynb"

active_run = mlflow.start_run(
    run_name="llm-train-plots-example-nb",
    tags=run_tags,
    log_system_metrics=True,
)
print("active run id:", active_run.info.run_id)

tracking.log_hardware(step=0)
tracking.log_params(run_params)

# Upload the notebook itself as a downloadable artifact.
if NOTEBOOK_PATH.exists():
    source_artifact_uri = tracking.save_data(NOTEBOOK_PATH, artifact_path=SOURCE_ARTIFACT_PATH)
    print("source artifact:", source_artifact_uri)
else:
    print("notebook path not found, skipping source upload:", NOTEBOOK_PATH)


## Step 4 — Simulate the training trajectory

`tracking.timed("simulate")` measures the block's wall-clock and logs it
as the `simulate_seconds` metric on the run.

In [ ]:
with tracking.timed("simulate"):
    data = simulate_training(
        total_steps=total_steps,
        warmup_steps=warmup_steps,
        eval_every=eval_every,
    )

print("series keys:", list(data.keys()))
print("steps:", data["steps"].shape, "eval_steps:", data["eval_steps"].shape)


## Step 5 — Build the history DataFrame and register it as an MLflow Dataset

An MLflow **Dataset** is a first-class entity (separate from artifacts) that
gets a row in the run's *Datasets* tab. Tying metrics to a dataset (via the
`dataset=` kwarg in `log_metrics`) lets the UI show "metric X on dataset Y".

In [ ]:
import pandas as pd

history_df = pd.DataFrame({
    "step": data["steps"],
    "train_loss": data["train_loss"],
    "train_ppl": data["train_ppl"],
    "learning_rate": data["learning_rate"],
    "grad_norm": data["grad_norm"],
    "tokens_per_sec": data["tokens_per_sec"],
    "step_seconds": data["step_seconds"],
})

train_dataset = tracking.log_dataset(
    history_df,
    name="pretrain-history",
    targets="train_loss",
    context="training",
)
history_df.head()


## Step 6 — Stream per-step metrics

This is what produces the time-series charts in the run page. Each call adds
one point at `step=N`; MLflow stitches them into a chart.

There's an `eval_*` set written less often (only at `eval_steps`), and a
training set written every step. This is the common pattern for pretraining.

In [ ]:
import time

with tracking.timed("log_step_metrics") as streaming_timer:
    sim_start = time.perf_counter()
    for i, step in enumerate(data["steps"].tolist()):
        tracking.log_metrics({
            "train_loss":     float(data["train_loss"][i]),
            "train_ppl":      float(data["train_ppl"][i]),
            "learning_rate":  float(data["learning_rate"][i]),
            "grad_norm":      float(data["grad_norm"][i]),
            "tokens_per_sec": float(data["tokens_per_sec"][i]),
            "step_seconds":   float(data["step_seconds"][i]),
        }, step=int(step))
    for j, step in enumerate(data["eval_steps"].tolist()):
        tracking.log_metrics({
            "eval_loss":      float(data["eval_loss"][j]),
            "eval_ppl":       float(data["eval_ppl"][j]),
            "eval_accuracy":  float(data["eval_accuracy"][j]),
            "eval_precision": float(data["eval_precision"][j]),
            "eval_recall":    float(data["eval_recall"][j]),
            "eval_f1":        float(data["eval_f1"][j]),
        }, step=int(step))
    tracking.log_duration("step_logging_wallclock", time.perf_counter() - sim_start)

print(f"streamed {total_steps} train steps + {len(data['eval_steps'])} eval steps "
      f"in {streaming_timer.elapsed:.1f}s")


## Step 7 — Train a real (small) sklearn model

The point isn't that this Ridge regressor is meaningful — it isn't. The
point is that we have *something concrete* to log + register, so the rest
of the workflow exercises the registry.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

with tracking.timed("fit_loss_predictor"):
    x_train = history_df[["step", "learning_rate"]].to_numpy()
    y_train = history_df["train_loss"].to_numpy()

    loss_predictor = Ridge(alpha=1.0, random_state=42)
    loss_predictor.fit(x_train, y_train)
    preds = loss_predictor.predict(x_train)

    tracking.log_params({
        "predictor_kind": "Ridge",
        "predictor_features": "step,learning_rate",
        "predictor_alpha": 1.0,
    })
    tracking.log_metrics({
        "predictor_mae": float(mean_absolute_error(y_train, preds)),
        "predictor_r2":  float(r2_score(y_train, preds)),
    }, dataset=train_dataset)

print("Ridge MAE:", mean_absolute_error(y_train, preds))
print("Ridge R^2:", r2_score(y_train, preds))


## Step 8 — Log AND register the model

`registered_model_name=...` is the bit that actually creates a version in
the Models tab. Without it, the model is only an artifact under this run.

`allow_registry_failure=True` downgrades transient registry-side errors to
a `RuntimeWarning` rather than failing the whole notebook — useful when
the tracking server is up but the registry backend is flaky.

In [ ]:
with tracking.timed("log_register_model"):
    tracking.log_model(
        loss_predictor,
        flavor="sklearn",
        artifact_path=MODEL_ARTIFACT_PATH,
        input_example=x_train[:3],
        params={"alpha": 1.0, "features": ["step", "learning_rate"]},
        registered_model_name=REGISTERED_MODEL_NAME,
        allow_registry_failure=True,
    )
print(f"logged + registered as: {REGISTERED_MODEL_NAME}")


## Step 9 — Build and log the plot bundle

`tracking.log_plot` saves each figure as a PNG artifact **and** logs it via
`mlflow.log_image` so it shows up in the run's Image Grid.

The `tracking.trace(...)` block wraps each plot in an MLflow span, so the
run also gets a trace timeline showing how long each plot took.

In [ ]:
from utils.plots import (
    plot_attention_heatmap, plot_eval_benchmarks, plot_gradient_norms,
    plot_learning_rate_schedule, plot_loss_curves, plot_perplexity,
    plot_step_time, plot_throughput, plot_token_length_distribution,
    plot_training_dashboard,
)


In [ ]:
with tracking.timed("plot_and_log"):
    with tracking.trace("plot_loss"):
        fig, _ = plot_loss_curves(
            train_loss=data["train_loss"], eval_loss=data["eval_loss"],
            steps=data["steps"], eval_steps=data["eval_steps"],
            title="Train vs Eval Loss",
        )
        tracking.log_plot(fig, key="loss_curves")

    with tracking.trace("plot_perplexity"):
        fig, _ = plot_perplexity(
            train_ppl=data["train_ppl"], eval_ppl=data["eval_ppl"],
            steps=data["steps"], eval_steps=data["eval_steps"],
        )
        tracking.log_plot(fig, key="perplexity")

    with tracking.trace("plot_lr"):
        fig, _ = plot_learning_rate_schedule(
            learning_rates=data["learning_rate"], steps=data["steps"],
        )
        tracking.log_plot(fig, key="learning_rate")

    with tracking.trace("plot_grad"):
        fig, _ = plot_gradient_norms(
            grad_norms=data["grad_norm"], steps=data["steps"], clip_threshold=1.0,
        )
        tracking.log_plot(fig, key="gradient_norms")

    with tracking.trace("plot_throughput"):
        fig, _ = plot_throughput(
            tokens_per_sec=data["tokens_per_sec"], steps=data["steps"],
        )
        tracking.log_plot(fig, key="throughput")

    with tracking.trace("plot_step_time"):
        fig, _ = plot_step_time(step_seconds=data["step_seconds"], steps=data["steps"])
        tracking.log_plot(fig, key="step_time")

    with tracking.trace("plot_dashboard"):
        fig, _ = plot_training_dashboard(
            history_df.drop(columns=["train_ppl"]),
            title="Training Dashboard",
        )
        tracking.log_plot(fig, key="training_dashboard")

    with tracking.trace("plot_token_lengths"):
        rng = np.random.default_rng(0)
        lengths = rng.integers(low=8, high=2048, size=20_000, endpoint=True)
        fig, _ = plot_token_length_distribution(lengths)
        tracking.log_plot(fig, key="token_length_distribution")

    with tracking.trace("plot_attention"):
        tokens = list("ABCDEFGHIJKLMNOP")
        fig, _ = plot_attention_heatmap(
            fake_attention_matrix(n_tokens=len(tokens)),
            tokens=tokens, title="Attention (synthetic)",
        )
        tracking.log_plot(fig, key="attention")

    with tracking.trace("plot_benchmarks"):
        scores = {"winogrande": 0.61, "arc_easy": 0.55, "hellaswag": 0.42,
                  "lambada": 0.38, "piqa": 0.66}
        baseline = {k: v - 0.04 for k, v in scores.items()}
        fig, _ = plot_eval_benchmarks(scores, baseline=baseline)
        tracking.log_plot(fig, key="eval_benchmarks")

print("plots uploaded")


## Step 10 — Final summary metrics + end-of-run hardware snapshot

Summary metrics (without a `step=`) appear in the run's *Overview* card —
they're what you sort and filter on in the experiments table.

A second `log_hardware(step=1)` paired with the `step=0` snapshot makes the
RAM-drift chart non-empty.

In [ ]:
tracking.log_metrics({
    "final_train_loss":   float(data["train_loss"][-1]),
    "final_eval_loss":    float(data["eval_loss"][-1]),
    "final_train_ppl":    float(data["train_ppl"][-1]),
    "final_eval_ppl":     float(data["eval_ppl"][-1]),
    "final_eval_f1":      float(data["eval_f1"][-1]),
    "best_eval_f1":       float(np.max(data["eval_f1"])),
    "mean_tokens_per_sec":float(np.mean(data["tokens_per_sec"])),
    "total_train_seconds":float(np.sum(data["step_seconds"])),
}, dataset=train_dataset)

tracking.log_hardware(step=1)


## Step 11 — Close the run

Forgetting `mlflow.end_run()` is the #1 notebook gotcha. The run will look
"running" forever in the UI and subsequent cells/notebooks will keep writing
into the wrong run. Always end it explicitly.

In [ ]:
mlflow.end_run()
print(f"run finished. registered model: {REGISTERED_MODEL_NAME} (alias: {MODEL_ALIAS})")


## Next

Try the GPU variant: **`example_training_plots_gpu.ipynb`** — same workflow
but the `Ridge` is swapped for a small PyTorch MLP trained on CUDA, logged
with the `pytorch` MLflow flavor.
